# Tools: od funkcji do kontekstu

1. Uruchamiamy pętlę narzędzi ręcznie, żeby zobaczyć decyzję modelu i wynik Pythona.
2. `@tool` i `create_agent` wykonują tę samą pętlę.
3. `args_schema` podaje modelowi jawny schemat i waliduje jego argumenty.
4. `ToolRuntime` daje narzędziu dane z `context`, a `Command` zapisuje wynik w `state`.

[Dokumentacja LangChain Tools](https://docs.langchain.com/oss/python/langchain/tools)


In [1]:
# Imports and project path setup only; examples start in the next cell.
from dataclasses import dataclass, replace
from datetime import datetime
from inspect import getdoc, signature
from pathlib import Path
import json
import sys
from typing import Literal

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.toml").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from langchain.agents import AgentState, create_agent
from langchain.tools import ToolRuntime, tool
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage, messages_to_dict
from langgraph.types import Command
from model_config import get_json_model, get_model, get_model_name
from pydantic import BaseModel, Field, ValidationError

In [2]:
model = get_model()
model_name = get_model_name()

MODEL [main] profile=local provider=ollama model=gemma4:12b


### 1. Manual tool loop

These are ordinary Python functions. First we write their tool schemas by hand. Then we build the same schemas automatically from function names, type hints, and docstrings. `bind_tools` sends a schema to the model; our Python loop still executes the chosen function.

In [3]:
def add(a: int, b: int) -> int:
    """Add two integers."""
    return a + b


def multiply(a: int, b: int) -> int:
    """Multiply two integers."""
    return a * b


TOOLS = {"add": add, "multiply": multiply}
MANUAL_TOOL_SCHEMAS = [
    {
        "type": "function",
        "function": {
            "name": "add",
            "description": "Add two integers.",
            "parameters": {
                "type": "object",
                "properties": {"a": {"type": "integer"}, "b": {"type": "integer"}},
                "required": ["a", "b"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "multiply",
            "description": "Multiply two integers.",
            "parameters": {
                "type": "object",
                "properties": {"a": {"type": "integer"}, "b": {"type": "integer"}},
                "required": ["a", "b"],
            },
        },
    },
]


def schema_from_function(function) -> dict:
    """Read this lesson's int arguments and description from a function."""
    properties = {}
    required = []
    for name, parameter in signature(function).parameters.items():
        if parameter.annotation is not int:
            raise TypeError(f"This example expects an int argument: {name}")
        properties[name] = {"type": "integer"}
        if parameter.default is parameter.empty:
            required.append(name)

    return {
        "type": "function",
        "function": {
            "name": function.__name__,
            "description": getdoc(function),
            "parameters": {
                "type": "object",
                "properties": properties,
                "required": required,
            },
        },
    }


AUTO_TOOL_SCHEMAS = [schema_from_function(function) for function in TOOLS.values()]
print("Hand-written schema:")
print(json.dumps(MANUAL_TOOL_SCHEMAS, indent=2))
print("Schema read from function signatures and docstrings:")
print(json.dumps(AUTO_TOOL_SCHEMAS, indent=2))

TOOL_SCHEMAS = MANUAL_TOOL_SCHEMAS  # Change to AUTO_TOOL_SCHEMAS to compare.
model_with_tools = model.bind_tools(TOOL_SCHEMAS)
print("Available tools:", list(TOOLS))

Hand-written schema:
[
  {
    "type": "function",
    "function": {
      "name": "add",
      "description": "Add two integers.",
      "parameters": {
        "type": "object",
        "properties": {
          "a": {
            "type": "integer"
          },
          "b": {
            "type": "integer"
          }
        },
        "required": [
          "a",
          "b"
        ]
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "multiply",
      "description": "Multiply two integers.",
      "parameters": {
        "type": "object",
        "properties": {
          "a": {
            "type": "integer"
          },
          "b": {
            "type": "integer"
          }
        },
        "required": [
          "a",
          "b"
        ]
      }
    }
  }
]
Schema read from function signatures and docstrings:
[
  {
    "type": "function",
    "function": {
      "name": "add",
      "description": "Add two integers.",
      "parameters": 

The loop below shows every step: user → model decision → Python tool → tool result → model again. The model may ask for several tools in one step. We stop when it answers without tool calls. The next model call includes the earlier tool request and its corresponding tool result.

In [4]:
NOTEBOOK_DIR = Path.cwd() if (Path.cwd() / "05_tools.ipynb").exists() else ROOT / "notebooks" / "langchain"
LOG_FILE = NOTEBOOK_DIR / "04_tools" / "execution_log.txt"
execution_log = []  # The latest run is also available in this notebook variable.


def ask_with_manual_tools(question: str) -> str:
    execution_log.clear()

    def record(text: str, show: bool = True) -> None:
        execution_log.append(text)
        if show:
            print(text)

    messages = [
        SystemMessage(content="Use the provided tools for arithmetic. Give a short final answer."),
        HumanMessage(content=question),
    ]
    record(f"RUN: {datetime.now().astimezone().isoformat(timespec='seconds')}")
    record(f"MODEL: {model_name}")
    record("TOOL SCHEMAS GIVEN TO MODEL:\n" + json.dumps(TOOL_SCHEMAS, indent=2), show=False)
    record(f"USER: {question}")

    try:
        for step in range(1, 6):
            record("-" * 60)
            record(f"STEP {step}: MODEL INPUT")
            for message in messages:
                record(f"  {message.type}: content={message.content!r}")
                if getattr(message, "tool_calls", None):
                    record(f"    earlier tool_calls={message.tool_calls}")
                if message.type == "tool":
                    record(f"    tool_call_id={message.tool_call_id}")
            record(
                "FULL MODEL INPUT JSON:\n" + json.dumps(messages_to_dict(messages), indent=2, ensure_ascii=False),
                show=False,
            )

            reply = model_with_tools.invoke(messages)
            record(
                "FULL MODEL REPLY JSON:\n" + json.dumps(messages_to_dict([reply]), indent=2, ensure_ascii=False),
                show=False,
            )
            messages.append(reply)  # Keep the model's tool request in the conversation.

            if reply.invalid_tool_calls:
                raise ValueError(f"Invalid tool call: {reply.invalid_tool_calls}")

            if not reply.tool_calls:
                record(f"MODEL ANSWER (step {step}): {reply.content}")
                return reply.content

            for call in reply.tool_calls:
                name, args = call["name"], call["args"]
                record(f"MODEL REQUEST (step {step}): {name}({args}); call_id={call['id']}")
                if name not in TOOLS:
                    raise ValueError(f"Unknown tool: {name}")

                result = TOOLS[name](**args)
                record(f"PYTHON EXECUTES: {name}(**{args}) -> {result}")
                messages.append(ToolMessage(content=str(result), tool_call_id=call["id"]))
                record(f"TOOL RESULT SENT TO MODEL: {result}")

        raise RuntimeError("The model did not finish within 5 steps.")
    except Exception as error:
        record(f"ERROR: {type(error).__name__}: {error}")
        raise
    finally:
        LOG_FILE.parent.mkdir(exist_ok=True)
        with LOG_FILE.open("a", encoding="utf-8") as output:
            output.write("\n".join(execution_log) + "\n\n")
        print(f"Saved execution log: {LOG_FILE}")

In [5]:
ask_with_manual_tools("What is (7 + 5) * 3? Use the tools and show the result.")

RUN: 2026-10-22T16:24:01+02:00
MODEL: gemma4:12b
USER: What is (7 + 5) * 3? Use the tools and show the result.
------------------------------------------------------------
STEP 1: MODEL INPUT
  system: content='Use the provided tools for arithmetic. Give a short final answer.'
  human: content='What is (7 + 5) * 3? Use the tools and show the result.'
MODEL REQUEST (step 1): add({'a': 7, 'b': 5}); call_id=949df4f4-8110-4082-b5d9-d1cbe271f2fe
PYTHON EXECUTES: add(**{'a': 7, 'b': 5}) -> 12
TOOL RESULT SENT TO MODEL: 12
------------------------------------------------------------
STEP 2: MODEL INPUT
  system: content='Use the provided tools for arithmetic. Give a short final answer.'
  human: content='What is (7 + 5) * 3? Use the tools and show the result.'
  ai: content=''
    earlier tool_calls=[{'name': 'add', 'args': {'a': 7, 'b': 5}, 'id': '949df4f4-8110-4082-b5d9-d1cbe271f2fe', 'type': 'tool_call'}]
  tool: content='12'
    tool_call_id=949df4f4-8110-4082-b5d9-d1cbe271f2fe
MODEL REQU

'The result of (7 + 5) * 3 is 36.'

### 2. LangChain agent with `@tool`

Now `@tool` builds the descriptions and argument schema from the functions. The next cell prints exactly what LangChain generated. `create_agent` performs the model/tool loop; `stream_mode="updates" shows the visible decisions, tool results, and final answer.

In [ ]:
@tool
def add_numbers(a: int, b: int) -> int:
    """Add two integers."""
    return a + b


@tool
def multiply_numbers(a: int, b: int) -> int:
    """Multiply two integers."""
    return a * b


native_tools = [add_numbers, multiply_numbers]
for selected_tool in native_tools:
    print(f"Python function: {selected_tool.name}{signature(selected_tool.func)}")
    print(f"Docstring read by @tool: {selected_tool.description}")
    print("LangChain input schema:")
    print(json.dumps(selected_tool.get_input_schema().model_json_schema(), indent=2))


agent = create_agent(
    model=model,
    tools=[add_numbers, multiply_numbers],
    system_prompt="Use the provided tools for arithmetic. Give a short final answer.",
)

Python function: add_numbers(a: int, b: int) -> int
Docstring read by @tool: Add two integers.
LangChain input schema:
{
  "description": "Add two integers.",
  "properties": {
    "a": {
      "title": "A",
      "type": "integer"
    },
    "b": {
      "title": "B",
      "type": "integer"
    }
  },
  "required": [
    "a",
    "b"
  ],
  "title": "add_numbers",
  "type": "object"
}
Python function: multiply_numbers(a: int, b: int) -> int
Docstring read by @tool: Multiply two integers.
LangChain input schema:
{
  "description": "Multiply two integers.",
  "properties": {
    "a": {
      "title": "A",
      "type": "integer"
    },
    "b": {
      "title": "B",
      "type": "integer"
    }
  },
  "required": [
    "a",
    "b"
  ],
  "title": "multiply_numbers",
  "type": "object"
}


In [ ]:
def ask_with_native_tools(question: str) -> None:
    print(f"USER: {question}")
    step = 0
    for update in agent.stream(
        {"messages": [{"role": "user", "content": question}]},
        stream_mode="updates",
    ):
        for node, state in update.items():
            step += 1
            print("-" * 60)
            print(f"STEP {step}: {node.upper()}")
            for message in state.get("messages", []):
                if message.type == "ai" and message.tool_calls:
                    for call in message.tool_calls:
                        print(f"MODEL DECISION: request {call['name']}({call['args']})")
                    if message.content:
                        print(f"MODEL CONTENT: {message.content}")
                elif message.type == "tool":
                    print(f"TOOL EXECUTED ({message.name}): {message.content}")
                elif message.type == "ai":
                    print(f"MODEL DECISION: final answer = {message.content}")
                else:
                    print(f"{node.upper()}: {message}")

In [ ]:
ask_with_native_tools("What is (7 + 5) * 3? Use the tools and show the result.")

In [ ]:
def short_ask_with_native_tools(question: str) -> str:
    result = agent.invoke({
        "messages": [{"role": "user", "content": question}]
    })

    answer = result["messages"][-1].content

    print(f"USER: {question}")
    print(f"ANSWER: {answer}")

    return answer


short_ask_with_native_tools("What is (7 + 5) * 3? Use the tools and show the result.")

### 3. Narzędzie z jawnym schematem argumentów

Wcześniej `@tool` odczytał typy z funkcji. Możemy też podać **własny schemat Pydantic** przez `args_schema`. Model dostaje nazwy i opisy pól, a Python sprawdza wartości, zanim uruchomi funkcję. W przykładzie obliczamy fikcyjną cenę dostawy; nie ma sklepu ani płatności.

`QuoteInput` wymaga odległości od 1 do 100 km i pozwala wybrać tylko `standard` albo `express`. Zobacz, że błędne dane nie przechodzą walidacji.


In [ ]:
class QuoteInput(BaseModel):
    distance_km: int = Field(ge=1, le=100, description="Odległość dostawy w kilometrach")
    speed: Literal["standard", "express"] = Field(
        default="standard", description="Sposób dostawy"
    )


@tool(args_schema=QuoteInput)
def base_delivery_price(distance_km: int, speed: str = "standard") -> int:
    """Oblicz demonstracyjną cenę dostawy w PLN."""
    return 10 + 2 * distance_km + (8 if speed == "express" else 0)


print("Schemat widoczny dla modelu:")
print(json.dumps(base_delivery_price.args_schema.model_json_schema(), ensure_ascii=False, indent=2))
print("Poprawne wywołanie przez Python:", base_delivery_price.invoke({"distance_km": 5, "speed": "express"}))
try:
    base_delivery_price.invoke({"distance_km": -5, "speed": "rocket"})
except ValidationError as error:
    print("Niepoprawne argumenty odrzucone:", error.error_count(), "błędy")


Agent używa tego samego schematu. Model może **poprosić** o `base_delivery_price`, ale nie wykonuje obliczenia sam. W `messages` zobaczymy osobno żądanie modelu, wynik funkcji Pythona i odpowiedź modelu.


In [ ]:
schema_agent = create_agent(
    model=model,
    tools=[base_delivery_price],
    system_prompt="Do wyceny dostawy zawsze użyj base_delivery_price. Odpowiedz krótko po polsku.",
)
schema_result = schema_agent.invoke({
    "messages": [{"role": "user", "content": "Ile kosztuje dostawa standard na 5 km?"}]
})
for message in schema_result["messages"]:
    if message.type == "ai" and message.tool_calls:
        print("MODEL PROSI:", message.tool_calls)
    elif message.type == "tool":
        print("PYTHON ZWRACA:", message.content)
print("ODPOWIEDŹ MODELU:", schema_result["messages"][-1].content)


### 4. Narzędzie czyta `context` i aktualizuje `state`

Teraz aplikacja przekaże `QuoteContext(user_id, discount_percent)` przy wywołaniu agenta. Ten kontekst jest **niewidoczny jako argument narzędzia dla modelu**. `ToolRuntime` dostarcza go funkcji Pythona. Wyobraź sobie, że rabat został wcześniej ustalony przez aplikację; w notebooku podajemy go ręcznie jako dane demonstracyjne.

Narzędzie zapisze obliczoną kwotę do `last_quote` w **stanie agenta** przez `Command`. W tym samym `Command` zwróci `ToolMessage`, żeby model zobaczył wynik obliczenia. `context` nie jest w ten sposób zmieniany.


In [ ]:
@dataclass(frozen=True)
class QuoteContext:
    user_id: str
    discount_percent: int


class QuoteState(AgentState):
    last_quote: int | None


TOOL_CONTEXT_SEEN = []  # Tylko lokalny ślad dydaktyczny; niczego nie zapisujemy.


@tool(args_schema=QuoteInput)
def quote_for_current_user(
    distance_km: int,
    runtime: ToolRuntime[QuoteContext, QuoteState],
    speed: str = "standard",
) -> Command:
    """Oblicz fikcyjną cenę dostawy z rabatem bieżącego użytkownika."""
    base = 10 + 2 * distance_km + (8 if speed == "express" else 0)
    discount = runtime.context.discount_percent
    price = base * (100 - discount) // 100
    TOOL_CONTEXT_SEEN.append((runtime.context.user_id, discount))
    return Command(update={
        "last_quote": price,
        "messages": [ToolMessage(
            content=f"Cena dostawy: {price} PLN.",
            tool_call_id=runtime.tool_call_id,
        )],
    })


print("Argumenty narzędzia widoczne dla modelu:", quote_for_current_user.args)


Aplikacja może podać **nowy obiekt `context` przy następnym wywołaniu**. W przykładzie zmienia rabat z 0% na 20%, pozostawiając `user_id`. Stary kontekst nadal ma 0%; narzędzie nie mutuje go podczas pracy. Każde wywołanie zaczyna osobny przebieg, bo nie dodajemy tu checkpointera.


In [ ]:
context_agent = create_agent(
    model=model,
    tools=[quote_for_current_user],
    context_schema=QuoteContext,
    state_schema=QuoteState,
    system_prompt="Do wyceny dostawy zawsze użyj quote_for_current_user. Odpowiedz krótko po polsku.",
)


def ask_for_quote(context: QuoteContext) -> dict:
    result = context_agent.invoke(
        {
            "messages": [{"role": "user", "content": "Ile kosztuje dostawa standard na 5 km?"}],
            "last_quote": None,
        },
        context=context,
    )
    for message in result["messages"]:
        if message.type == "ai" and message.tool_calls:
            print("MODEL PROSI:", [call["name"] for call in message.tool_calls])
        elif message.type == "tool":
            print("PYTHON ZWRACA:", message.content)
    print("STATE last_quote:", result["last_quote"])
    print("ODPOWIEDŹ MODELU:", result["messages"][-1].content)
    return result


first_context = QuoteContext(user_id="u-101", discount_percent=0)
first_quote = ask_for_quote(first_context)
updated_context = replace(first_context, discount_percent=20)
second_quote = ask_for_quote(updated_context)
print("Stary context:", first_context)
print("Nowy context na kolejne wywołanie:", updated_context)
print("ID i rabat odczytane wyłącznie przez Python:", TOOL_CONTEXT_SEEN)


| Element | Skąd pochodzi? | Czy model go podaje? | Co się zmienia? |
|---|---|---|---|
| `distance_km`, `speed` | żądanie narzędzia | tak, zgodnie ze schematem | argumenty pojedynczego wywołania narzędzia |
| `user_id`, `discount_percent` | `context` od aplikacji | nie | aplikacja tworzy nowy kontekst przy kolejnym `invoke` |
| `last_quote` | wynik obliczenia w Pythonie | nie | narzędzie aktualizuje `state` przez `Command` |
| odpowiedź dla użytkownika | model po wyniku narzędzia | tak | komunikat końcowy |

**Najważniejsze:** `context` jest wejściem bieżącego przebiegu. Gdy narzędzie ma zapisać wynik *w trakcie* przebiegu, aktualizuje `state`, nie `runtime.context`. Kod pokazuje fikcyjne wyceny i lokalny ślad; nie wywołuje bazy ani zewnętrznego systemu.
